# Earnings-Call Sentiment â€” run on Colab T4

Runs Phase A (LLM parser) and Phase B (LLM extractor) on earnings-call transcripts using **qwen3:14b** via Ollama on a Colab T4 GPU (15 GB VRAM).

**Before running:** Runtime â†’ Change runtime type â†’ **T4 GPU**.

**Test mode:** by default this notebook processes only the **first 3 transcripts** as a smoke test (~5â€“10 min total). Once that works end-to-end, set `N_LIMIT = None` in the config cell to run all 131.

Pipeline:
1. Install Ollama on Colab and start the daemon.
2. Pull `qwen3:14b` (~9 GB).
3. Get the project source (git clone OR upload).
4. Upload `ECT.zip` (the 131 transcripts).
5. Run parser â†’ `data/parsed/*.json`.
6. Run extractor â†’ `data/extractions/*.json`.
7. Zip results and download.

Cached outputs live under `data/parsed/` and `data/extractions/`. If a cell crashes, just re-run it â€” already-processed transcripts will be skipped.

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Install Ollama and start the daemon

Colab is Linux, so the official installer works. We start `ollama serve` as a background process and give it a moment to come up.

In [ ]:
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess, time, os

# Force GPU usage
os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"

# Start Ollama server in the background
ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(5)  # let it bind
print(f"Ollama daemon PID: {ollama_proc.pid}")
!ollama --version

## 3. Pull qwen3:14b

~9 GB download. Takes 2â€“5 minutes on Colab's network.

In [ ]:
!ollama pull qwen3:14b

In [ ]:
!ollama list

## 4. Get the project source

**Option A** â€” clone from your GitHub repo (recommended; cleanest):

In [ ]:
!git clone https://github.com/liron1996/EarningsCallSentiment.git
%cd EarningsCallSentiment

_Option B (upload zip) is no longer needed — `git clone` above brings the source AND the 131 transcripts in `ECT/`. Skip the next two cells (the upload-zip cell and the ECT.zip upload cell)._

In [ ]:
# Skipped — using git clone (cell 10) instead of zip upload.
import os
print('cwd:', os.getcwd())
print('Project files present:', sorted(f for f in os.listdir('.') if not f.startswith('.'))[:10])

## 5. Install Python dependencies

In [ ]:
!pip install -e . --quiet

## 6. Verify transcripts are present

`git clone` already brought `ECT/` with all 131 transcripts. This cell just sanity-checks the count.

In [ ]:
import os
txt_files = [f for f in os.listdir('ECT') if f.endswith('.txt')]
print(f"ECT/ contains {len(txt_files)} .txt files")
assert len(txt_files) == 131, f"Expected 131 transcripts, got {len(txt_files)}"

## 7. Configure for Colab T4 + qwen3:14b

Override the local-machine defaults via env vars. Must be set **before** importing the project modules.

**`N_LIMIT`** controls how many transcripts to process:
- `3` (default) â†’ smoke-test on first 3 files (~5â€“10 min). Use this first.
- `None` â†’ run all 131 (~3 hours). Switch to this after the smoke test passes.

In [ ]:
import os

# === Test mode â€” set to None to process all 131 transcripts ===
N_LIMIT = 3

os.environ['ECS_MODEL']            = 'qwen3:14b'   # 9 GB; fits in T4's 15 GB VRAM
os.environ['ECS_NUM_CTX_MAX']      = '32768'       # 32k ctx (KV cache fits with 14b)
os.environ['ECS_NUM_PREDICT']      = '4096'        # bigger output budget for richer JSON
os.environ['ECS_MAX_CHUNK_CHARS']  = '80000'       # most files fit in 1 chunk; only the largest split
os.environ['ECS_KEEP_ALIVE']       = '60m'         # keep model resident across the whole batch
print({k: v for k, v in os.environ.items() if k.startswith('ECS_')})

# Pick the subset of transcripts to process (alphabetical, deterministic).
all_txt = sorted(f for f in os.listdir('ECT') if f.endswith('.txt'))
selected_txt = all_txt if N_LIMIT is None else all_txt[:N_LIMIT]
selected_json = [f[:-4] + '.json' for f in selected_txt]
print(f"\nN_LIMIT = {N_LIMIT}")
print(f"Will process {len(selected_txt)} transcript(s): {selected_txt}")

## 8. Run Phase A â€” parser

~30â€“90 seconds per transcript on T4. Idempotent (cached); safe to re-run if interrupted.

In [ ]:
files_arg = ' '.join(selected_txt)
!python -m earnings_call_sentiment.parse --files {files_arg}

In [ ]:
import json, glob
summary = json.load(open('data/parsed/_summary.json'))
print('Status counts:', summary['by_status'])
print('Files in data/parsed/:', len(glob.glob('data/parsed/*.json')) - 1)  # minus _summary.json

## 9. Run Phase B â€” sentiment + event extractor

Reads each `data/parsed/*.json` and produces `data/extractions/*.json`. ~30â€“60s per file. Cached.

In [ ]:
files_arg = ' '.join(selected_json)
!python -m earnings_call_sentiment.extract --files {files_arg}

In [ ]:
import json, glob
summary = json.load(open('data/extractions/_summary.json'))
print('Status counts:', summary['by_status'])
print('Files in data/extractions/:', len(glob.glob('data/extractions/*.json')) - 1)

## 10. Spot-check one extraction

In [ ]:
import json
sample_name = selected_json[0]
print(f"Inspecting: {sample_name}\n")
d = json.load(open(f'data/extractions/{sample_name}'))
for k in ('overall_tone', 'tone_bucket', 'guidance', 'themes', 'wins', 'risks'):
    print(f"{k}: {d.get(k)}")

## 11. Bundle & download results

Zips `data/parsed/` + `data/extractions/` and triggers a browser download.

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('data_outputs', 'zip', 'data')
files.download('data_outputs.zip')